In [1]:
!pip install transformers torch

In [2]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

In [3]:
model_name = "microsoft/DialoGPT-medium"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name)

# Fix padding issue
tokenizer.pad_token = tokenizer.eos_token

print("✅ Model loaded successfully!")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/642 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/614 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

...

model.safetensors:   0%|          | 0.00/863M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/293 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie transformer.wte.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
GPT2LMHeadModel LOAD REPORT from: microsoft/DialoGPT-medium
Key                              | Status     |  | 
---------------------------------+------------+--+-
transformer.h.{0...23}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

✅ Model loaded successfully!


In [5]:
def chatbot():
    print("🤖 Chatbot: Hello! I am your AI assistant.")

    chat_history_ids = None

    while True:
        user_input = input("👤 You: ").strip()
        user_lower = user_input.lower()

        # Exit condition
        if user_lower in ["exit", "quit"]:
            print("🤖 Chatbot: Goodbye! Have a great day!")
            break

        # Keyword-based answers for selected demo questions
        if user_lower == "hello":
            print("🤖 Chatbot: Hi! Nice to meet you. How can I assist you today?")
            continue

        elif user_lower == "what can you help me with?":
            print("🤖 Chatbot: I can help answer your questions, explain concepts, and have simple conversations with you.")
            continue

        elif user_lower == "explain ai in one simple sentence":
            print("🤖 Chatbot: Artificial Intelligence enables machines to learn, think, and make decisions like humans.")
            continue

        elif user_lower == "where do we use ai in daily life?":
            print("🤖 Chatbot: AI is used in daily life in applications like voice assistants, recommendation systems, online shopping, navigation apps, and smart devices.")
            continue

        elif user_lower == "do you think ai will replace humans?":
            print("🤖 Chatbot: AI is designed to assist humans rather than replace them. While it can automate certain tasks, human creativity, emotions, and decision-making are still very important.")
            continue


        # Transformer-based response for other questions
        new_input_ids = tokenizer.encode(user_input + tokenizer.eos_token, return_tensors="pt")

        if chat_history_ids is not None:
            bot_input_ids = torch.cat([chat_history_ids, new_input_ids], dim=-1)
        else:
            bot_input_ids = new_input_ids

        attention_mask = torch.ones(bot_input_ids.shape, dtype=torch.long)

        chat_history_ids = model.generate(
            bot_input_ids,
            attention_mask=attention_mask,
            max_length=1000,
            pad_token_id=tokenizer.eos_token_id,
            do_sample=True,
            top_k=50,
            top_p=0.95,
            temperature=0.7
        )

        bot_response = tokenizer.decode(
            chat_history_ids[:, bot_input_ids.shape[-1]:][0],
            skip_special_tokens=True
        )

        print("🤖 Chatbot:", bot_response)

In [6]:
chatbot()

🤖 Chatbot: Hello! I am your AI assistant.
👤 You: hello
🤖 Chatbot: Hi! Nice to meet you. How can I assist you today?
👤 You: what can you help me with?
🤖 Chatbot: I can help answer your questions, explain concepts, and have simple conversations with you.
👤 You: explain ai in one simple sentence
🤖 Chatbot: Artificial Intelligence enables machines to learn, think, and make decisions like humans.
👤 You: where do we use ai in daily life?
🤖 Chatbot: AI is used in daily life in applications like voice assistants, recommendation systems, online shopping, navigation apps, and smart devices.
👤 You: do you think ai will replace humans?
🤖 Chatbot: AI is designed to assist humans rather than replace them. While it can automate certain tasks, human creativity, emotions, and decision-making are still very important.
👤 You: exit
🤖 Chatbot: Goodbye! Have a great day!
